In [1]:
import os
print(os.listdir('Data'))

['cleaned_ecommerce_master_sales.csv', 'customer_master.csv', 'dataset_statistics.csv', 'ecommerce_sales_customer_analytics_150k.csv', 'order_items.csv', 'product_catalog.csv']


In [2]:
# ---- Cell 2 code ----
import pandas as pd
import numpy as np

# Load each raw file from the Data/ folder
customers = pd.read_csv('Data/customer_master.csv')
items     = pd.read_csv('Data/order_items.csv')
products  = pd.read_csv('Data/product_catalog.csv')
sales     = pd.read_csv('Data/ecommerce_sales_customer_analytics_150k.csv')

raw_tables = {'customers': customers, 'items': items, 'products': products, 'sales': sales}

# Baseline data quality: shape and null counts per table
for name, df in raw_tables.items():
    print(f'--- {name} | shape: {df.shape} ---')
    print(df.isna().sum()[df.isna().sum() > 0])  # show only columns that have nulls

--- customers | shape: (25000, 11) ---
Series([], dtype: int64)
--- items | shape: (397569, 12) ---
Series([], dtype: int64)
--- products | shape: (1175, 9) ---
Series([], dtype: int64)
--- sales | shape: (138116, 46) ---
delivery_days               24557
estimated_delivery_days     24557
return_status              128654
return_reason              128654
customer_rating             24557
review_sentiment            24557
customer_review             24557
campaign_name               83233
coupon_code                110502
dtype: int64


In [3]:
# ---- Cell 3 code ----
# Make column names consistent (lowercase, underscores) so keys match across tables
for df in raw_tables.values():
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

id_cols = ['customer_id', 'order_id', 'product_id']

for name, df in raw_tables.items():
    for col in id_cols:
        if col in df.columns:  # only touch ID columns that exist in this table
            # Cast to text, trim spaces, drop float artifacts like "101.0"
            df[col] = df[col].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
            df[col] = df[col].replace('nan', np.nan)  # restore true nulls for later filling
    print(name, {c: str(df[c].dtype) for c in id_cols if c in df.columns})

customers {'customer_id': 'object'}
items {'order_id': 'object', 'product_id': 'object'}
products {'product_id': 'object'}
sales {'customer_id': 'object', 'order_id': 'object'}


In [4]:
# ---- Cell 4 code ----
def clean_table(df):
    df = df.copy()  # work on a copy so the raw data stays untouched

    # Convert date columns first, so they are not filled with text later
    for col in [c for c in df.columns if 'date' in c]:
        df[col] = pd.to_datetime(df[col], errors='coerce')  # bad values become NaT

    num_cols = df.select_dtypes(include='number').columns
    txt_cols = df.select_dtypes(include='object').columns

    df[num_cols] = df[num_cols].fillna(0)               # missing numbers become 0
    df[txt_cols] = df[txt_cols].fillna('Not Specified') # missing text becomes a label
    return df

customers, items, products, sales = [clean_table(raw_tables[k]) for k in ['customers', 'items', 'products', 'sales']]

# Confirm no nulls remain (dates may keep NaT on purpose)
print(sales.isna().sum().sum(), customers.isna().sum().sum())

0 0


In [5]:
# ---- Cell 5 code ----
def add_details(left, right, key, check):
    # Bring in only new columns from the right table to avoid duplicate column names
    new_cols = [key] + [c for c in right.columns if c not in left.columns]
    # how='left' keeps every transaction; validate catches accidental row duplication
    return left.merge(right[new_cols], on=key, how='left', validate=check)

master = add_details(sales, items, 'order_id', 'one_to_many')          # order line items
master = add_details(master, customers, 'customer_id', 'many_to_one')  # customer details
master = add_details(master, products, 'product_id', 'many_to_one')    # product details

master = clean_table(master)  # refill nulls from unmatched left-join rows

print('Sales rows:', len(sales), '| Master rows:', len(master))  # row growth = multi-item orders
print(master.shape)

Sales rows: 138116 | Master rows: 397569
(397569, 56)


In [6]:
# ---- Cell 6 code ----
# index=False keeps the pandas row numbers out of the CSV
master.to_csv('Data/cleaned_ecommerce_master_sales.csv', index=False)
print('Saved:', master.shape)

Saved: (397569, 56)


In [7]:
# Point df at the sales table explicitly (headers and IDs already standardized, nulls not yet filled)
# Why: without this line, df only exists because of a leftover loop variable from cell 3
df = raw_tables['sales']

# len(df) returns the number of rows; f"{:,}" adds thousands separators for readability
print(f"Rows in df: {len(df):,}")

# .shape returns (rows, columns) if you also want the column count
print(f"Shape (rows, columns): {df.shape}")



Rows in df: 138,116
Shape (rows, columns): (138116, 46)


In [8]:
# Validate df against the raw file: real zeros must stay 0 and true missing values must stay NULL
raw = pd.read_csv('Data/ecommerce_sales_customer_analytics_150k.csv')

# Apply the same header cleanup so column names match df
raw.columns = raw.columns.str.strip().str.lower().str.replace(' ', '_')

# Safety check: df and raw must be the same rows in the same order
# Why: we copy values by position, so a mismatch would put values on the wrong orders
assert len(raw) == len(df), "Row counts differ - do not copy"
assert (raw['net_sales'].fillna(0).values == df['net_sales'].values).all(), "Row order differs - do not copy"

# These two columns contain real zeros (e.g. same-day delivery) that must stay 0, not NULL
cols_to_restore = ['delivery_days', 'estimated_delivery_days']

# .values copies by position: this guarantees df matches the raw file for these two columns
df[cols_to_restore] = raw[cols_to_restore].values

# Verify: zeros should be 6,674 and 5,794; nulls should be 24,557 each
print(df[cols_to_restore].eq(0).sum())
print(df[cols_to_restore].isna().sum())

delivery_days              6674
estimated_delivery_days    5794
dtype: int64
delivery_days              24557
estimated_delivery_days    24557
dtype: int64


In [9]:
# ---- Cell 9 code ----
# Export a 1,000-row preview so reviewers can see the cleaned master table without a 200 MB download
sample = master.sample(1000, random_state=42).copy()  # fixed random_state returns the same rows on every run

# Safety check before export: same columns as master, exactly 1,000 rows
assert list(sample.columns) == list(master.columns), "Column mismatch"
assert len(sample) == 1000, "Unexpected sample size"

sample.to_csv('Data/cleaned_ecommerce_master_sales_sample.csv', index=False)  # index=False keeps pandas row numbers out
print(f"--- sample | shape: {sample.shape} ---")
print(f"Sample rows: {len(sample):,} | Master rows: {len(master):,}")

--- sample | shape: (1000, 56) ---
Sample rows: 1,000 | Master rows: 397,569
